# AbLang -- Antibody Language Model

![AbLang -- Antibody Language Model](https://proto-bio.github.io/proto-assets/images/tool/ablang/hero.png)

This notebook demonstrates how to use **AbLang** for antibody-specific embeddings, sequence scoring, and masked position restoration.

## What is AbLang?

AbLang is a family of antibody-specific language models trained on antibody sequence data from the Observed Antibody Space (OAS). It provides:

- **Embeddings** -- Antibody representations for similarity search, clustering, and downstream ML
- **Scoring** -- Pseudo-log-likelihood scoring to assess antibody sequence naturalness
- **Restore** -- Predict masked positions in antibody sequences (e.g., CDR loop reconstruction)

**Links:** [Paper (DOI: 10.1093/bioadv/vbae040)](https://doi.org/10.1093/bioadv/vbae040) | [GitHub](https://github.com/oxpig/AbLang2)

## Model Variants

AbLang wraps two generations of models under a single tool. The `model_choice` config field controls which variant is used:

| `model_choice` | Model | Input Format | Embedding Dim | When to Use |
|---|---|---|---|---|
| `"ablang1-heavy"` | AbLang1 heavy | Single heavy chain | 768 | VH-only analysis |
| `"ablang1-light"` | AbLang1 light | Single light chain | 768 | VL-only analysis |
| `"ablang2-paired"` | AbLang2 | Paired `"heavy\|light"` | 480 | Paired VH+VL analysis |

## Automatic Model Routing

By default, `model_choice="auto"`. The tool inspects the input sequences and picks the right model:

- If **any** sequence contains a `|` separator → routes to **`ablang2-paired`**
- Otherwise → routes to **`ablang1-heavy`** (the most common single-chain use case)

To use `ablang1-light`, you must set `model_choice="ablang1-light"` explicitly.

## Imports

In [1]:
from proto_tools.tools.masked_models.ablang import (
    run_ablang_embeddings,
    AbLangEmbeddingsInput,
    AbLangEmbeddingsConfig,
    run_ablang_score,
    AbLangScoringInput,
    AbLangScoringConfig,
    run_ablang_sample,
    AbLangSampleInput,
    AbLangSampleConfig,
    run_ablang_gradient,
    AbLangGradientInput,
    AbLangGradientConfig,
)
from proto_tools.entities.antibody import Antibody, AntibodyLogits
from proto_tools.utils import one_hot_protein_logits
from proto_tools.utils.notebook_docs import display_api_reference

## Input API Reference

The model variant is auto-resolved from the chains present on the `Antibody` / `AntibodyLogits` object: heavy-only -> `ablang1-heavy`, light-only -> `ablang1-light`, both -> `ablang2-paired`.

### `run_ablang_embeddings`

In [2]:
display_api_reference("ablang", "input", "run_ablang_embeddings")

**Input** — `AbLangEmbeddingsInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>antibodies</code> | <code>list[Antibody]</code> | required | Antibody sequence(s) to embed. |

### `run_ablang_score`

In [3]:
display_api_reference("ablang", "input", "run_ablang_score")

**Input** — `AbLangScoringInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>antibodies</code> | <code>list[Antibody]</code> | required | Antibody sequence(s) to score. |

### `run_ablang_sample`

In [4]:
display_api_reference("ablang", "input", "run_ablang_sample")

**Input** — `AbLangSampleInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>antibodies</code> | <code>list[Antibody]</code> | required | Antibody sequence(s) with '_' at positions to restore. |

### `run_ablang_gradient`

In [5]:
display_api_reference("ablang", "input", "run_ablang_gradient")

**Input** — `AbLangGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>antibody</code> | <code>AntibodyLogits</code> | required | Antibody with relaxed sequence distributions over amino acids. |
| <code>temperature</code> | <code>float &#124; None</code> | <code>None</code> | Softmax temperature. Applies softmax(input / T) when set. |

## Config API Reference

### `run_ablang_embeddings`

In [6]:
display_api_reference("ablang", "config", "run_ablang_embeddings")

**Config** — `AbLangEmbeddingsConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position amino-acid logits in the output (large; disable to save memory) |

### `run_ablang_score`

In [7]:
display_api_reference("ablang", "config", "run_ablang_score")

**Config** — `AbLangScoringConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |
| <code>scoring_mode</code> | <code>Literal['pseudo_log_likelihood', 'confidence']</code> | <code>'pseudo_log_likelihood'</code> | Per-position masked PLL (accurate, O(L) passes) vs single-pass confidence proxy |

### `run_ablang_sample`

In [8]:
display_api_reference("ablang", "config", "run_ablang_sample")

**Config** — `AbLangSampleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |
| <code>temperature</code> | <code>float</code> | <code>1.0</code> | Softmax temperature for amino-acid sampling. 0 = greedy argmax (ablang restore); >0 = stochastic. |
| <code>align</code> | <code>bool</code> | <code>False</code> | Run ANARCI alignment first; enables extension of unknown-length termini |
| <code>return_logits</code> | <code>bool</code> | <code>False</code> | Include per-position logits in the output (large; disable to save memory) |

### `run_ablang_gradient`

In [9]:
display_api_reference("ablang", "config", "run_ablang_gradient")

**Config** — `AbLangGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>use_ste</code> | <code>bool</code> | <code>False</code> | Hard one-hot forward pass with soft-probability gradients |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only log-likelihood |
| <code>batch_size</code> | <code>int &#124; None</code> | <code>None</code> | AA positions per forward pass. Lower if OOM, higher for throughput |

## Output API Reference

### `run_ablang_embeddings`

In [10]:
display_api_reference("ablang", "output", "run_ablang_embeddings")

**Output** — `AbLangEmbeddingsOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[SequenceEmbedding]</code> | required | Per-sequence embedding results |

**`SequenceEmbedding`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>mean_embedding</code> | <code>list[float]</code> | required | Mean-pooled embedding vector (averaged over sequence length) |
| <code>attention_mask</code> | <code>list[int]</code> | required | Binary mask: 1 = valid position, 0 = padding |
| <code>logits</code> | <code>list[list[float]] &#124; None</code> | <code>None</code> | Per-position amino acid logits (seq_len, vocab_size) |
| <code>projection</code> | <code>Projection2D &#124; None</code> | <code>None</code> | 2D UMAP projection of this sequence's embedding within the call's batch |

**`Projection2D`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>x</code> | <code>float</code> | required | First reduced coordinate |
| <code>y</code> | <code>float</code> | required | Second reduced coordinate |

### `run_ablang_score`

In [11]:
display_api_reference("ablang", "output", "run_ablang_score")

**Output** — `MaskedModelScoringOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>scores</code> | <code>list[MaskedModelScoringMetrics]</code> | required | List of scoring outputs, one per input sequence |

**Metrics** — `MaskedModelScoringMetrics` (one set per `scores` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>avg_log_likelihood</code> | <code>float</code> | <code>&lt;= 0</code> |  |  |
| <code>perplexity</code> **(primary)** | <code>float</code> | <code>&gt;= 1</code> |  |  |

### `run_ablang_sample`

In [12]:
display_api_reference("ablang", "output", "run_ablang_sample")

**Output** — `AbLangSampleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | Sampled/restored protein sequences |
| <code>logits</code> | <code>list[list[list[float]]] &#124; None</code> | <code>None</code> | Per-position amino acid logits. Shape: [num_sequences, seq_len, 20]. |

## 1. Extract Embeddings (auto-routing)

With `model_choice="auto"` (the default), the tool detects single-chain vs paired sequences automatically. Here we pass single heavy chains, so the tool routes to `ablang1-heavy` and returns 768-dim embeddings.

In [13]:
# Trastuzumab (Herceptin) VH sequence
trastuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"

# Pertuzumab VH sequence (another anti-HER2 antibody)
pertuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFTFTDYTMDWVRQAPGKGLEWVADVNPNSGGSIYNQRFKGRFTLSVDRSKNTLYLQMNSLRAEDTAVYYCARNLGPSFYFDYWGQGTLVTVSS"

# Single-chain Antibody objects auto-route to ablang1-heavy
inputs = AbLangEmbeddingsInput(
    antibodies=[Antibody(heavy_chain=trastuzumab_vh), Antibody(heavy_chain=pertuzumab_vh)]
)
config = AbLangEmbeddingsConfig(batch_size=2)

embeddings_result = run_ablang_embeddings(inputs, config)

print(f"Model used: {embeddings_result.metadata['model_choice']}")
print(f"Processed {len(embeddings_result.results)} sequences")
print(f"Embedding dimension: {len(embeddings_result.results[0].mean_embedding)}")
print(f"Attention mask length (trastuzumab): {sum(embeddings_result.results[0].attention_mask)}")
print(f"Attention mask length (pertuzumab): {sum(embeddings_result.results[1].attention_mask)}")

Running run_ablang_embeddings [00:00]

Model used: ablang1-heavy
Processed 2 sequences
Embedding dimension: 768
Attention mask length (trastuzumab): 120
Attention mask length (pertuzumab): 119


## 2. Score Sequences

Compute pseudo-log-likelihood scores to assess how natural an antibody sequence looks to the model. Lower (more negative) scores indicate lower model confidence. This is useful for comparing natural antibodies against engineered or scrambled variants.

Again using auto-routing here — single chains route to `ablang1-heavy`.

In [14]:
# Compare a natural antibody heavy chain vs a scrambled version
natural_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
# Scrambled version (same amino acids, random order — no non-standard residues)
scrambled_vh = "GVSTYGDRGYPQFSVWVDTAQYNRGISGLKRAAKEGSTYTIRRFLPVDSANDWTYGLPHVSGMTEQNLVCSAYGIKARLWPGNVTSELDFAGTKYREPCGQIHALNMSSWGTDAVYAEV"

# Single-chain Antibody objects auto-route to ablang1-heavy
inputs = AbLangScoringInput(
    antibodies=[Antibody(heavy_chain=natural_vh), Antibody(heavy_chain=scrambled_vh)]
)
config = AbLangScoringConfig(
    scoring_mode="pseudo_log_likelihood",
    batch_size=2,
)

score_result = run_ablang_score(inputs, config)

print("Average log-likelihood (higher = more natural):")
print(f"  Natural VH:   {score_result.scores[0].avg_log_likelihood:.3f}")
print(f"  Scrambled VH: {score_result.scores[1].avg_log_likelihood:.3f}")
print("Perplexity (lower = more natural):")
print(f"  Natural VH:   {score_result.scores[0].perplexity:.3f}")
print(f"  Scrambled VH: {score_result.scores[1].perplexity:.3f}")

Running run_ablang_score [00:00]

Average log-likelihood (higher = more natural):
  Natural VH:   -0.823
  Scrambled VH: -3.206
Perplexity (lower = more natural):
  Natural VH:   2.276
  Scrambled VH: 24.669


## 3. Restore Masked Positions

Use AbLang to predict amino acids at masked positions (marked with `_`). This is especially useful for restoring CDR3 regions or filling in missing residues in antibody sequences.

In [15]:
# Trastuzumab VH with CDR3 region masked (positions 99-110: SRWGGDGFYAMDY)
masked_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYC_____________WGQGTLVTVSS"

print(f"Original CDR3: SRWGGDGFYAMDY")
print(f"Masked input:  ...TAVYYC_____________WGQGTL...")

# Single-chain Antibody auto-routes to ablang1-heavy
inputs = AbLangSampleInput(antibodies=[Antibody(heavy_chain=masked_vh)])
sample_result = run_ablang_sample(inputs)

restored_seq = sample_result.sequences[0]
# Extract the restored CDR3 region
restored_cdr3 = restored_seq[97:110]
print(f"Restored CDR3: {restored_cdr3}")
print(f"Full restored: {restored_seq}")

Original CDR3: SRWGGDGFYAMDY
Masked input:  ...TAVYYC_____________WGQGTL...


Running run_ablang_sample [00:00]

Restored CDR3: SHLRSDGAYVDSW
Full restored: EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCASHLRSDGAYVDSWGQGTLVTVSS


## 4. Paired Heavy|Light Mode (auto-routing to ablang2-paired)

Set both `heavy_chain` and `light_chain` on a single `Antibody` and the tool auto-routes to `ablang2-paired`, which processes heavy and light chains together to capture inter-chain dependencies. The paired model produces 480-dim embeddings instead of 768-dim. Sample outputs for paired antibodies are returned as a single string `"heavy|light"` you can split on `|` to recover the two chains.

In [16]:
# Trastuzumab paired heavy|light chains
trastuzumab_vh = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
trastuzumab_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"

# Setting both chains on a single Antibody auto-routes to ablang2-paired
paired_ab = Antibody(heavy_chain=trastuzumab_vh, light_chain=trastuzumab_vl)

inputs = AbLangEmbeddingsInput(antibodies=[paired_ab])
paired_result = run_ablang_embeddings(inputs)
print(f"Model used: {paired_result.metadata['model_choice']}")
print(f"Paired embedding dimension: {len(paired_result.results[0].mean_embedding)}")

# Scoring — also auto-routes to ablang2-paired
inputs = AbLangScoringInput(antibodies=[paired_ab])
config = AbLangScoringConfig(scoring_mode="pseudo_log_likelihood")
paired_score = run_ablang_score(inputs, config)
print(f"Paired log-likelihood: {paired_score.scores[0].avg_log_likelihood:.3f}")

# Restore masked positions in paired mode (mask CDR3 of light chain)
masked_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYC_________FGQGTKVEIK"
inputs = AbLangSampleInput(antibodies=[Antibody(heavy_chain=trastuzumab_vh, light_chain=masked_vl)])
paired_restore = run_ablang_sample(inputs)
restored_heavy, restored_light = paired_restore.sequences[0].split("|")
print(f"Original VL CDR3: QQHYTTPPT")
print(f"Restored VL:      {restored_light}")

Running run_ablang_embeddings [00:00]

Model used: ablang2-paired
Paired embedding dimension: 480


Running run_ablang_score [00:00]

Paired log-likelihood: -0.934


Running run_ablang_sample [00:00]

Original VL CDR3: QQHYTTPPT
Restored VL:      DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCLHADSFQQTFGQGTKVEIK


## 5. Light-Chain Routing

A light-chain-only `Antibody` (only `light_chain` set) auto-routes to `ablang1-light` — no extra setting is needed.

In [17]:
# Light-only Antibody (no heavy chain) auto-routes to ablang1-light
trastuzumab_vl = "DIQMTQSPSSLSASVGDRVTITCRASQDVNTAVAWYQQKPGKAPKLLIYSASFLYSGVPSRFSGSRSGTDFTLTISSLQPEDFATYYCQQHYTTPPTFGQGTKVEIK"

inputs = AbLangEmbeddingsInput(antibodies=[Antibody(light_chain=trastuzumab_vl)])

light_result = run_ablang_embeddings(inputs)

print(f"Model used: {light_result.metadata['model_choice']}")
print(f"Embedding dimension: {len(light_result.results[0].mean_embedding)}")

Running run_ablang_embeddings [00:00]

Model used: ablang1-light
Embedding dimension: 768


## 6. Compute Gradients

`run_ablang_gradient` differentiates AbLang's masked pseudo-log-likelihood objective with respect to a relaxed `(L, 20)` distribution over amino acids (wrapped in an `AntibodyLogits` entity, which can carry a heavy chain, a light chain, or both). Use this as a differentiable, antibody-specific pLM loss inside MCMC or gradient descent over relaxed sequences. Set `compute_gradient=False` for forward-only PLL scoring without the backward pass.

In [18]:
# Build relaxed logits for both chains. one_hot_protein_logits with sharpness=2.0
# yields a biased-but-not-saturated softmax target — a reasonable seed for MCMC.
heavy_logits = one_hot_protein_logits(trastuzumab_vh, sharpness=2.0)
light_logits = one_hot_protein_logits(trastuzumab_vl, sharpness=2.0)
antibody = AntibodyLogits(heavy_chain=heavy_logits, light_chain=light_logits)

gradient_input = AbLangGradientInput(antibody=antibody)
gradient_config = AbLangGradientConfig(
    use_ste=False,           # Soft blended embeddings
    compute_gradient=True,   # Set False for forward-only scoring
    device="cuda",           # Change to "cpu" if no GPU available
)

gradient_result = run_ablang_gradient(gradient_input, gradient_config)

print(f"Mean NLL:           {gradient_result.loss:.3f}")
print(f"Log-likelihood:     {gradient_result.metrics['log_likelihood']:.3f}")
print(f"Sequence length:    {gradient_result.metrics['sequence_length']}")
print(f"Model choice:       {gradient_result.metrics['model_choice']}")
assert gradient_result.gradient is not None
print(f"Gradient shape:     ({len(gradient_result.gradient)}, {len(gradient_result.gradient[0])})")
print(f"Gradient row 0 (first 5 cols): {[round(v, 4) for v in gradient_result.gradient[0][:5]]}")

Running run_ablang_gradient [00:00]

Mean NLL:           0.926
Log-likelihood:     -210.147
Sequence length:    227
Model choice:       ablang2-paired
Gradient shape:     (227, 20)
Gradient row 0 (first 5 cols): [0.017, 0.062, -0.0017, 0.0051, 0.0219]


## 7. Export Results

Save results to files for downstream analysis.

### Supported export formats

| Output type | Formats |
|---|---|
| Embeddings | `csv`, `json`, `npy`, `pt` |
| Scoring | `csv`, `json` |
| Sampling | `fasta`, `txt`, `json` |

In [19]:
# Export embeddings to CSV
embeddings_result.export("ablang_embeddings", export_path="./example_output", file_format="csv")
print("Embeddings exported to ./example_output/ablang_embeddings.csv")

# Export scores to JSON
score_result.export("ablang_scores", export_path="./example_output", file_format="json")
print("Scores exported to ./example_output/ablang_scores.json")

# Export restored sequences to FASTA
sample_result.export("ablang_restored", export_path="./example_output", file_format="fasta")
print("Restored sequences exported to ./example_output/ablang_restored.fasta")

Embeddings exported to ./example_output/ablang_embeddings.csv
Scores exported to ./example_output/ablang_scores.json
Restored sequences exported to ./example_output/ablang_restored.fasta
